# N1 - 2º Bimestre - Individual - Aprofundamento Estatístico

## Aluno: Guilherme Nascimento  
## Tema: Diferença estatística do desgaste da ferramenta entre máquinas com falha e sem falha

**Disciplina:** Ciência de Dados  
**Curso:** Engenharia de Controle e Automação  
**Projeto:** Manutenção Preditiva de "Zero-Downtime"  
**Dataset:** AI4I 2020 Predictive Maintenance Dataset

## 1. Objetivo

Este notebook tem como objetivo investigar se existe diferença estatisticamente significativa na variável **`tool_wear_[min]`** entre máquinas com falha e sem falha, com base no dataset **AI4I 2020 Predictive Maintenance Dataset**.

A proposta busca aprofundar um dos achados observados na etapa de Análise Exploratória de Dados (EDA), utilizando conceitos de Estatística Inferencial para verificar se o padrão identificado pode ser considerado relevante para o contexto de manutenção preditiva.

## 2. Carregamento do Dataset

Nesta etapa, carregamos a base tratada utilizada no projeto para realizar a análise estatística individual.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import shapiro, mannwhitneyu, ttest_ind

plt.rcParams["figure.figsize"] = (10, 6)
sns.set_style("whitegrid")

In [ ]:
df = pd.read_csv("../../data/processed/dados_tratados.csv")
df.head()

## 3. Seleção da Variável

A variável escolhida para esta análise foi **`tool_wear_[min]`**, por representar uma medida diretamente relacionada ao desgaste operacional do equipamento e por ter apresentado indícios de associação com a variável de falha durante a etapa exploratória do projeto.

In [ ]:
variavel = "tool_wear_[min]"
alvo = "machine_failure"

df[[variavel, alvo]].head()

## 4. Separação dos Grupos

Para a comparação estatística, os dados foram divididos em dois grupos:

- máquinas **sem falha** (`machine_failure = 0`)
- máquinas **com falha** (`machine_failure = 1`)

In [ ]:
grupo_sem_falha = df[df["machine_failure"] == 0][variavel]
grupo_com_falha = df[df["machine_failure"] == 1][variavel]

print("Quantidade sem falha:", len(grupo_sem_falha))
print("Quantidade com falha:", len(grupo_com_falha))

## 5. Estatística Descritiva

Antes da aplicação dos testes estatísticos, é importante observar medidas descritivas dos grupos, como média, mediana, desvio padrão e quartis.

In [ ]:
resumo = pd.DataFrame({
    "Sem falha": grupo_sem_falha.describe(),
    "Com falha": grupo_com_falha.describe()
})

resumo

In [ ]:
sns.boxplot(data=df, x="machine_failure", y=variavel)
plt.title("Desgaste da ferramenta por ocorrência de falha")
plt.xlabel("Falha da máquina")
plt.ylabel("Desgaste da ferramenta [min]")
plt.show()

### Interpretação inicial

A análise descritiva e a visualização por boxplot permitem observar, de forma preliminar, se há diferença de comportamento da variável `tool_wear_[min]` entre os grupos com e sem falha. Essa etapa não confirma significância estatística, mas orienta a formulação das hipóteses e a escolha do teste adequado.

## 6. Teste de Normalidade

Antes de definir o teste estatístico principal, foi aplicado o teste de **Shapiro-Wilk** para verificar se os dados seguem distribuição normal em cada grupo.

In [ ]:
amostra_sem_falha = grupo_sem_falha.sample(min(5000, len(grupo_sem_falha)), random_state=42)
amostra_com_falha = grupo_com_falha.sample(min(5000, len(grupo_com_falha)), random_state=42)

shapiro_sem = shapiro(amostra_sem_falha)
shapiro_com = shapiro(amostra_com_falha)

print("Shapiro - grupo sem falha:")
print(f"Estatística = {shapiro_sem.statistic:.4f}, p-valor = {shapiro_sem.pvalue:.6f}")

print("\nShapiro - grupo com falha:")
print(f"Estatística = {shapiro_com.statistic:.4f}, p-valor = {shapiro_com.pvalue:.6f}")

### Interpretação do teste de normalidade

- Se o **p-valor > 0,05**, não há evidência suficiente para rejeitar a hipótese de normalidade.
- Se o **p-valor ≤ 0,05**, considera-se que os dados não seguem distribuição normal.

A partir desse resultado, será escolhido o teste estatístico mais adequado para comparar os grupos.

## 7. Hipótese Nula e Hipótese Alternativa

Para esta análise, foram formuladas as seguintes hipóteses:

- **Hipótese Nula (H0):** não existe diferença estatisticamente significativa nos valores de `tool_wear_[min]` entre máquinas com falha e sem falha.
- **Hipótese Alternativa (H1):** existe diferença estatisticamente significativa nos valores de `tool_wear_[min]` entre máquinas com falha e sem falha.

## 8. Escolha do Teste Estatístico

A escolha do teste foi baseada no resultado do teste de normalidade:

- caso ambos os grupos apresentem distribuição aproximadamente normal, será utilizado o **teste t para amostras independentes**;
- caso pelo menos um dos grupos não apresente normalidade, será utilizado o **teste de Mann-Whitney U**, mais adequado para distribuições não normais.

Como a variável é numérica e a comparação ocorre entre dois grupos independentes, essas são as abordagens mais apropriadas para o problema.

## 9. Execução do Teste Estatístico

In [ ]:
if shapiro_sem.pvalue > 0.05 and shapiro_com.pvalue > 0.05:
    teste_nome = "Teste t para amostras independentes"
    teste = ttest_ind(grupo_sem_falha, grupo_com_falha, equal_var=False)
    estatistica = teste.statistic
    p_valor = teste.pvalue
else:
    teste_nome = "Teste de Mann-Whitney U"
    teste = mannwhitneyu(grupo_sem_falha, grupo_com_falha, alternative="two-sided")
    estatistica = teste.statistic
    p_valor = teste.pvalue

print("Teste escolhido:", teste_nome)
print(f"Estatística do teste = {estatistica:.4f}")
print(f"p-valor = {p_valor:.6f}")

### Interpretação do p-valor

Adotando nível de significância de **5% (0,05)**:

- se **p-valor < 0,05**, rejeita-se a hipótese nula;
- se **p-valor ≥ 0,05**, não se rejeita a hipótese nula.

Esse resultado indicará se a diferença observada entre os grupos pode ser considerada estatisticamente significativa.

## 10. Tamanho do Efeito

Além do p-valor, foi calculado o **tamanho do efeito** para avaliar a magnitude prática da diferença observada entre os grupos. Para isso, foi utilizada a métrica **Cohen's d**.

In [ ]:
media_sem = grupo_sem_falha.mean()
media_com = grupo_com_falha.mean()

std_sem = grupo_sem_falha.std()
std_com = grupo_com_falha.std()

n_sem = len(grupo_sem_falha)
n_com = len(grupo_com_falha)

desvio_pooled = np.sqrt(((n_sem - 1) * std_sem**2 + (n_com - 1) * std_com**2) / (n_sem + n_com - 2))
cohens_d = (media_sem - media_com) / desvio_pooled

print(f"Cohen's d = {cohens_d:.4f}")

### Interpretação do tamanho do efeito

Como referência geral para o **Cohen's d**:

- **0,2** → efeito pequeno
- **0,5** → efeito médio
- **0,8 ou mais** → efeito grande

Essa medida complementa o p-valor, mostrando se a diferença entre os grupos possui relevância prática além da significância estatística.

Embora o teste principal adotado tenha sido o Mann-Whitney U, o Cohen’s d foi utilizado de forma complementar para oferecer uma noção da magnitude prática da diferença observada entre os grupos.

## 11. Conclusão

Com base nos resultados obtidos, conclui-se que existe **diferença estatisticamente significativa** nos valores de **`tool_wear_[min]`** entre máquinas com falha e sem falha. O teste de normalidade de Shapiro-Wilk indicou que os dados não seguem distribuição normal nos grupos analisados, justificando a utilização do **teste de Mann-Whitney U**. O resultado do teste apresentou **p-valor < 0,001**, valor inferior ao nível de significância de 5% (0,05), o que leva à **rejeição da hipótese nula (H0)**. Isso indica que a diferença observada no desgaste da ferramenta entre os grupos dificilmente ocorreu ao acaso.

Além disso, a análise descritiva mostrou que as máquinas com falha apresentam valores de desgaste da ferramenta superiores aos observados em máquinas sem falha. Dessa forma, os resultados sugerem que o aumento do desgaste da ferramenta está associado à ocorrência de falhas, o que é coerente com o contexto de manutenção preditiva.

Assim, conclui-se que a variável **`tool_wear_[min]`** apresenta evidência estatística de associação com a ocorrência de falhas, podendo ser considerada uma variável relevante para as próximas etapas de seleção de atributos e modelagem preditiva. 